# 🧠 Aula 04 — Fundamentos de Processos e Threads

**Objetivo:** distinguir **processos** e **threads** em ambientes CPU e GPU, e aplicar
esse conhecimento para projetar pipelines de IA eficientes, paralelos e escaláveis.

**Roteiro deste notebook:**
1. Verificação do ambiente (núcleos da CPU e GPU).
2. Teoria: processos vs. threads e o GIL.
3. Demo: threading vs. multiprocessing numa tarefa CPU-bound.
4. Demo: threading numa tarefa I/O-bound.
5. Atividade: kernels CUDA com diferentes blocos/threads.
6. Discussão e síntese.

> 💡 **Sem GPU?** O notebook detecta e explica o conceito de warps/blocos com os números
> de referência — a aula roda do começo ao fim.

## 1. Verificação do Ambiente

Quantos núcleos de CPU temos (paralelismo real do host) e há GPU NVIDIA (para os kernels)?

In [ ]:
# @title 🔍 Núcleos da CPU e presença de GPU
# ============================================================================
# OBJETIVO: saber quantos núcleos temos (para o multiprocessing) e se há GPU.
# ============================================================================
import os, shutil

print(f"Núcleos lógicos de CPU: {os.cpu_count()}")

CAMINHO_NVIDIA_SMI = shutil.which("nvidia-smi")
TEM_GPU = CAMINHO_NVIDIA_SMI is not None
if TEM_GPU:
    import subprocess
    info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout
    print(f"GPU detectada: {info.strip()}")
else:
    print("Sem GPU NVIDIA — os kernels CUDA usarão os números de referência.")
    print("Para GPU real: Runtime ➔ Change runtime type ➔ T4 GPU.")

## 2. Teoria: processos, threads e o GIL

| Critério | Processo | Thread |
| :--- | :--- | :--- |
| Memória | **Isolada** por processo | **Compartilhada** |
| Criação | Lenta (*fork*) | Rápida |
| Comunicação | IPC (overhead) | Direta (risco de *race condition*) |
| Falha isolada | Sim | Não (derruba o pai) |
| GIL Python | Contorna | Afeta em CPU-bound |
| Melhor para | **CPU-bound** | **I/O-bound** |

**O que é o GIL?** O *Global Interpreter Lock* é um mutex que deixa **apenas uma thread
Python** executar bytecode por vez — mesmo em CPUs multi-core. Por isso `threading` **não**
acelera tarefas CPU-bound. Use `multiprocessing` para isso. NumPy e PyTorch **liberam o
GIL** nas operações nativas, e por isso escalam bem.

## 3. Demo: threading vs. multiprocessing (CPU-bound)

Vamos medir a **mesma tarefa pesada de CPU** (contar primos) de três formas. O resultado
expõe o GIL: threading empata com o sequencial; multiprocessing acelera de verdade.

In [ ]:
# @title ⏱️ CPU-bound: sequencial vs. threading vs. multiprocessing
# ============================================================================
# OBJETIVO: mostrar que threading NÃO acelera tarefa CPU-bound (GIL), mas
# multiprocessing sim (cada tarefa em um processo separado).
# ============================================================================
import math, multiprocessing, threading, time

LIMITE = 500_000   # até que número procurar primos (pesado p/ medir)
N_TAREFAS = 4

def calcular_primos(limite):
    """Tarefa CPU-bound: conta quantos primos existem até `limite`."""
    primos = 0
    for n in range(2, limite):
        if all(n % i != 0 for i in range(2, int(math.sqrt(n)) + 1)):
            primos += 1
    return primos

# ── 1) Sequencial ──────────────────────────────────────────────────────────
inicio = time.perf_counter()
for _ in range(N_TAREFAS):
    calcular_primos(LIMITE)
t_seq = time.perf_counter() - inicio
print(f"Sequencial      : {t_seq:6.2f}s")

# ── 2) Threading (o GIL limita) ────────────────────────────────────────────
inicio = time.perf_counter()
threads = [threading.Thread(target=calcular_primos, args=(LIMITE,)) for _ in range(N_TAREFAS)]
for t in threads: t.start()
for t in threads: t.join()
t_thr = time.perf_counter() - inicio
print(f"Threading       : {t_thr:6.2f}s  (GIL -> quase sem ganho)")

# ── 3) Multiprocessing (contorna o GIL) ────────────────────────────────────
inicio = time.perf_counter()
with multiprocessing.Pool(processes=N_TAREFAS) as pool:
    pool.starmap(calcular_primos, [(LIMITE,)] * N_TAREFAS)
t_mp = time.perf_counter() - inicio
print(f"Multiprocessing : {t_mp:6.2f}s  (speedup real)")
print(f"\nSpeedup Multiprocessing vs. sequencial: {t_seq / t_mp:.2f}x")

## 4. Demo: threading numa tarefa I/O-bound

Em tarefas de **I/O** (espera de rede/disco), o Python **libera o GIL** durante a espera.
Agora as threads esperam **em paralelo** — ganho real. Aqui simulamos a latência com
`time.sleep` (que também representa espera, não cálculo).

In [ ]:
# @title ⏱️ I/O-bound: sequencial vs. threading
# ============================================================================
# OBJETIVO: mostrar que threading AJUDA quando a tarefa é de I/O (o GIL é
# liberado durante a espera).
# ============================================================================
import threading, time

LATENCIA = 0.5   # segundos por 'requisição'
N_TAREFAS = 6

def requisicao_simulada(identificador, resultados):
    """Simula uma chamada de rede: espera e registra o resultado."""
    time.sleep(LATENCIA)       # espera de I/O (libera o GIL)
    resultados.append(identificador)

# ── 1) Sequencial ──────────────────────────────────────────────────────────
resultados = []
inicio = time.perf_counter()
for i in range(N_TAREFAS):
    requisicao_simulada(i, resultados)
t_seq = time.perf_counter() - inicio
print(f"Sequencial (I/O): {t_seq:6.2f}s")

# ── 2) Threading ───────────────────────────────────────────────────────────
resultados = []
inicio = time.perf_counter()
threads = [threading.Thread(target=requisicao_simulada, args=(i, resultados)) for i in range(N_TAREFAS)]
for t in threads: t.start()
for t in threads: t.join()
t_thr = time.perf_counter() - inicio
print(f"Threading  (I/O): {t_thr:6.2f}s  <- quase {N_TAREFAS}x mais rapido")
print(f"\nSpeedup com threads: {t_seq / t_thr:.1f}x")

## 5. Atividade: kernels CUDA — blocos e threads

Agora o modelo da GPU. Variamos `threads_por_bloco` (32 a 1024) numa soma de vetores e
medimos o tempo; depois usamos um **grid 2D** para processar uma imagem.

> ⚠️ **Só funciona com GPU NVIDIA (CUDA/Numba).** Sem GPU, a célula mostra os números de
> referência de uma Tesla T4.

In [ ]:
# @title 🧩 CUDA: variar threads por bloco (Numba)
# ============================================================================
# OBJETIVO: entender a hierarquia Thread -> Warp -> Bloco -> Grid medindo o
# efeito de threads_por_bloco no tempo de um kernel de soma de vetores.
# ============================================================================
try:
    from numba import cuda
    import numpy as np, time
    if not cuda.is_available():
        raise RuntimeError("sem GPU CUDA")

    N = 1024 * 1024   # 1M elementos
    a = np.ones(N, dtype=np.float32); b = np.ones(N, dtype=np.float32)
    c = np.zeros(N, dtype=np.float32)
    a_d = cuda.to_device(a); b_d = cuda.to_device(b); c_d = cuda.to_device(c)

    @cuda.jit
    def soma_vetores(a, b, c):
        idx = cuda.grid(1)                 # índice GLOBAL da thread
        if idx < a.shape[0]:
            c[idx] = a[idx] + b[idx]

    print(f"{'threads/bloco':>13} | {'blocos':>8} | {'tempo (ms)':>10}")
    print('-' * 40)
    for threads_por_bloco in (32, 64, 128, 256, 512, 1024):
        blocos = (N + threads_por_bloco - 1) // threads_por_bloco
        soma_vetores[blocos, threads_por_bloco](a_d, b_d, c_d)  # warm-up
        cuda.synchronize()
        inicio = time.perf_counter()
        for _ in range(50):
            soma_vetores[blocos, threads_por_bloco](a_d, b_d, c_d)
        cuda.synchronize()
        tempo = (time.perf_counter() - inicio) / 50
        print(f"{threads_por_bloco:>13} | {blocos:>8} | {tempo * 1000:>10.3f}")
    print('-' * 40)
    print("256 threads/bloco costuma ser o ótimo (múltiplo de 32 = 1 warp).")
except Exception as erro:
    print(f"Numba/CUDA indisponível aqui ({erro}).")
    print("Números de referência (Tesla T4): 32->0.320ms, 64->0.180ms, 128->0.095ms,")
    print("256->0.088ms (ótimo), 512->0.101ms, 1024->0.115ms.")

In [ ]:
# @title 🖼️ CUDA: grid 2D para processar uma imagem
# ============================================================================
# OBJETIVO: usar um grid 2D (x = coluna, y = linha) em que cada thread
# converte 1 pixel para escala de cinza — o mapeamento natural p/ imagens.
# ============================================================================
try:
    from numba import cuda
    import numpy as np
    if not cuda.is_available():
        raise RuntimeError("sem GPU CUDA")

    @cuda.jit
    def escala_cinza(img_rgb, img_gray):
        x, y = cuda.grid(2)             # x = coluna, y = linha
        if x < img_rgb.shape[1] and y < img_rgb.shape[0]:
            r = img_rgb[y, x, 0]; g = img_rgb[y, x, 1]; b = img_rgb[y, x, 2]
            img_gray[y, x] = np.uint8(0.299 * r + 0.587 * g + 0.114 * b)

    H, W = 1080, 1920
    img = np.random.randint(0, 256, (H, W, 3), dtype=np.uint8)
    gray = np.zeros((H, W), dtype=np.uint8)
    img_d = cuda.to_device(img); gray_d = cuda.to_device(gray)

    BLOCO = (16, 16)                    # 256 threads por bloco, em 2D
    GRID = ((W + 15) // 16, (H + 15) // 16)
    print(f"Grid : {GRID[0]} x {GRID[1]} blocos")
    print(f"Bloco: {BLOCO[0]} x {BLOCO[1]} threads")
    print(f"Total de threads: {GRID[0] * GRID[1] * BLOCO[0] * BLOCO[1]:,}")

    escala_cinza[GRID, BLOCO](img_d, gray_d)
    cuda.synchronize()
    resultado = gray_d.copy_to_host()
    print(f"Imagem {H}x{W} convertida para cinza! (shape {resultado.shape})")
except Exception as erro:
    print(f"Numba/CUDA indisponível aqui ({erro}).")
    print("Conceito: grid 2D -> cada thread (x,y) processa 1 pixel da imagem.")

## 6. Discussão em Grupo

Em grupos de 3–4, com base no cenário da startup:

1. O pré-processamento de imagens (CPU) é um gargalo. Threading ou multiprocessing?
2. Se um warp tem 32 threads e 16 entram no `if` e 16 no `else`, como a GPU executa?
3. Por que o PyTorch usa múltiplos **processos** (workers) no DataLoader?
4. Qual a vantagem de 256 threads/bloco em vez de 1024?

> Atividade de pesquisa completa em `aulas/aula04/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Processo ou thread?** Para cada tarefa, escolha e justifique em uma frase:
*(a)* baixar 20 arquivos de um servidor; *(b)* calcular os primos de 4 faixas grandes em
paralelo; *(c)* manter a interface de um app respondendo enquanto salva um arquivo.

**2) O GIL.** Explique, com suas palavras, por que o `threading` **não** acelera tarefas
CPU-bound em Python, mas NumPy/PyTorch escalam bem.

**3) Medindo o speedup.** Na célula-esqueleto, rode multiprocessing com 1, 2 e 4 processos e
registre o tempo e o speedup. O speedup acompanha o número de processos? Por que não
exatamente?

**4) Hierarquia da GPU.** Ordene e explique o papel de: **Thread, Warp, Bloco, Grid**. Por que
o warp tem exatamente 32 threads em lockstep (SIMD)?

**5) Divergência de warp.** Se 16 threads de um warp entram no `if` e 16 no `else`, como a
GPU executa isso? Qual o impacto no desempenho e como evitá-lo?


In [ ]:
# @title Exercício 3 — speedup do multiprocessing
# ============================================================================
# OBJETIVO: medir o speedup variando o numero de processos.
# ============================================================================
import math, multiprocessing, time

def calcular_primos(limite):
    primos = 0
    for n in range(2, limite):
        if all(n % i != 0 for i in range(2, int(math.sqrt(n)) + 1)):
            primos += 1
    return primos

LIMITE = 400_000
TAREFAS = 4

# Referencia sequencial
inicio = time.perf_counter()
for _ in range(TAREFAS):
    calcular_primos(LIMITE)
t_seq = time.perf_counter() - inicio
print(f"Sequencial: {t_seq:.2f}s")

print(f"\n{'processos':>10} | {'tempo (s)':>10} | {'speedup':>8}")
print("-" * 34)
for nproc in (1, 2, 4):
    inicio = time.perf_counter()
    with multiprocessing.Pool(processes=nproc) as pool:
        pool.starmap(calcular_primos, [(LIMITE,)] * TAREFAS)
    t = time.perf_counter() - inicio
    print(f"{nproc:>10} | {t:>10.2f} | {t_seq / t:>7.2f}x")

# Pergunta: o speedup acompanha o numero de processos? O que limita?

## 8. Síntese e Tarefa de Casa

**O que levar:**
- **Processo:** memória isolada, contorna o GIL, ideal para **CPU-bound**.
- **Thread (CPU):** memória compartilhada, GIL limita CPU-bound, ideal para **I/O-bound**.
- **Thread (GPU):** uma instância do kernel, registradores próprios.
- **Warp:** 32 threads SIMD; divergência = perda de desempenho.
- **Bloco:** grupo de warps, shared memory, 1 SM (máx. 1024 threads).
- **Grade:** conjunto de blocos = o problema completo.

**Tarefa (opcional):** implemente um pipeline paralelo de pré-processamento com
`multiprocessing.Pool`:
- 100 imagens sintéticas 512×512;
- conversão para escala de cinza;
- compare o tempo sequencial vs. 2, 4 e 8 processos;
- plote o gráfico de *speedup* com Matplotlib.

> 🔗 **Próxima aula:** *Redes e Transferência de Dados* — o dataset pré-processado agora
> precisa vir de um storage remoto de 200 GB, atravessando a rede.